# Using (online) API
    
Alternatively, instead of using your local resources, various companies provide __APIs__ for their proprietary models. Note that the owner of the key pays on a token basis. The prices shown here were checked on 19.5.2026 and may change. 


| Model | Provider | Input  | Output  |  
| -------- | ------- | :--------: | :--------: |
|     |  | per 1M tokens | per 1M tokens | 
| GPT-5.4  |  OpenAI    | \$ 2.50 | \$ 15.00 |  
| GPT-5.4 mini |  OpenAI    | \$ 0.75 | \$ 4.50 | 
| GPT-5.4 nano |  OpenAI    | \$ 0.20 | \$ 1.25 | 
| GPT 5.4-pro   | OpenAI | \$30.00 | \$180.00 |
| Gemini Pro 3.1    | Google | \$ 2.00 - 4.00 | \$ 12.00 - 18.00 |
|  Gemini Flash 3.1   | Google | \$ 0.50 | \$ 3.00 |
| Claude Opus 4.7 | Anthropic | \$ 6.25  | \$ 25.00 |
| Claude Sonnet 4.6 | Anthropic | \$ 3.00 | \$ 15.00 |
<!-- > 
price of Opus 4.7 is now smaller than for 4.0; 
all other models became more more expensive 
<-->



* Gemini Pro: price increases for inputs > __200k tokens__
* OpenAI offers a (more expensive) __real-time__ API and (cheaper) __batched API__ (may take up to 24h, running asynchronously)
* Depending on the provided requests per minute (RPM) and tokens per minute (TPM) might be limited according to the type of contract/subscription.  
* While there are different providers, the following example is using the OpenAI interface 
* To run the code, an api key is needed; this can be done either via writing it into a file
* __NEVER__ upload api key to __public repository__! $\Rightarrow$ Anybody with the key can use the API at your cost.
* [keys](https://platform.openai.com/api-keys), [usage](https://platform.openai.com/usage) and the model's requests can be monitored online. 
* The code is part of the [paper](https://doi.org/10.21203/rs.3.rs-6566994/v1) _Creation, Evaluation and Self-Validation of Simulation Models with Large Language Models_, see also our [github](https://github.com/jgerstmayr/AI-engineering-lab) and the file [processRessultsWithOpenAI_GPT4o](https://github.com/jgerstmayr/AI-engineering-lab/blob/main/processResultsWithOpenAI_GPT4o.py).
* Good approach fr creting simulation code:
    1. Provide problem description and let model choose simulation/model elements
    2. Provide example code for chosen elements and create code
__Talk__ 
<!-- 
Send out file with API keys for workshop.
-->

## Note: 
* To run the following script, an API key is required!
* You do not need to create one yourself
* One key will be sent out shortly before the workshop and deactivated afterwards. 

In [1]:
import os
import time
from LLMHelperFunctions import CheckOutputLLM # helper function
import openai
from openai import OpenAI
from dotenv import load_dotenv

flag_env = load_dotenv("openai_api_key.env")
if flag_env: 
    print('API-key loaded successfully')
else: 
    print('The API-key could not be loaded - check if it is available!')
    print('A key will be provided in the workshop.')

API-key loaded successfully


Initialize the API using the (private) key. The key is a combination of alphanumeric characters similar to an ssh-key.   
It could also be hardcoded as a string in the interface below. The key is created [online](https://platform.openai.com/api-keys).  

In [2]:
# initialize the client using the API key from the environment variable
client = OpenAI(
        api_key= os.environ.get("OPENAI_API_KEY", None),
    )

### Availible models
A list of availibe models is also availible [online](https://platform.openai.com/docs/models/).  
Some models (e.g. 4o) have different input/output modalities. 
The date (e.g. ) refers to the date of the _snapshot_. If the behaviour/performance of a specific model should be consistant the snapshot can be specified. 



In [3]:
modelList = client.models.list().to_dict()
print('model list: ')
for i, data in enumerate(modelList['data']): 
    name_i = data['id']
    print(i, ': ', data['id'])

model list: 
0 :  text-embedding-ada-002
1 :  whisper-1
2 :  gpt-3.5-turbo
3 :  tts-1
4 :  gpt-3.5-turbo-16k
5 :  gpt-4-0613
6 :  gpt-4
7 :  davinci-002
8 :  babbage-002
9 :  gpt-3.5-turbo-instruct
10 :  gpt-3.5-turbo-instruct-0914
11 :  gpt-3.5-turbo-1106
12 :  tts-1-hd
13 :  tts-1-1106
14 :  tts-1-hd-1106
15 :  text-embedding-3-small
16 :  text-embedding-3-large
17 :  gpt-3.5-turbo-0125
18 :  gpt-4-turbo
19 :  gpt-4-turbo-2024-04-09
20 :  gpt-4o
21 :  gpt-4o-2024-05-13
22 :  gpt-4o-mini-2024-07-18
23 :  gpt-4o-mini
24 :  gpt-4o-2024-08-06
25 :  omni-moderation-latest
26 :  omni-moderation-2024-09-26
27 :  o1-2024-12-17
28 :  o1
29 :  o3-mini
30 :  o3-mini-2025-01-31
31 :  gpt-4o-2024-11-20
32 :  gpt-4o-mini-search-preview-2025-03-11
33 :  gpt-4o-mini-search-preview
34 :  gpt-4o-transcribe
35 :  gpt-4o-mini-transcribe
36 :  o1-pro-2025-03-19
37 :  o1-pro
38 :  gpt-4o-mini-tts
39 :  o3-2025-04-16
40 :  o4-mini-2025-04-16
41 :  o3
42 :  o4-mini
43 :  gpt-4.1-2025-04-14
44 :  gpt-4.1
45 

In [4]:
dt_total, nTokensTotal = 0, 0
with open("ContextGeneral.txt", "r") as file:
    # Read all lines into a list
    lines = file.readlines()
with open("promptSliderCrank.txt", "r") as file:
    myPrompt = file.read()
    
myContext  = "".join(lines[31:])
print('the loaded contest is:\n', '*'*40, '\n' ,myContext)

print('\n'*2, '*'*40, 'The Prompt is:',  '*'*40, myPrompt)

the loaded contest is:
 **************************************** 
 #In the following, there are examples to create multibody systems in Exudyn and also important notes about the interface.
#NOTE: mbs.Create...(...) calls several functions in the background to create nodes, objects, markers and loads in Exudyn.
#most quantities such as initial or reference positions and velocities are giving as 3D lists [x,y,z] for positions, velocities, ....
#rotations are usually given as rotation matrix (3x3 numpy array); 
#RotationVector2RotationMatrix([rotX, rotY, rotZ]) computes a rotation around the global x,y,z rotation axis
#for working with rigid bodies, note that there is always a local coordinate system in the body, 
#which can be used to define the location of position and orientation of joints, see the examples.
    
#%%++++++++++++++++++++++++++++++++++++++++++++++++++++
#create rigid bodies and mass points with distance constraint and joints
import exudyn as exu
from exudyn.utilities imp

## Context
As previously mentioned, LLMs learn through context which enables the user to provide information inside the prompt without requiring fine-tuning the model parameters. In the Multibody strategies for this are laid out e.g. in  
* Möltner, T., Manzl, P., Pieber, M., & Gerstmayr, J. (2025). Creation, evaluation and self-validation of simulation models with large language models. Neurocomputing. [doi: 10.1016/j.neucom.2025.132030](https://doi.org/10.1016/j.neucom.2025.132030)


While this example uses the simulation tool Exudyn, context could also be used to contain information on other tools like coding Chrono, MBDyn, or FEniCS. 



### Create Output
Start inference with one of the shown models.  
Here, the context can be given either as a system prompt or as part of the content.  

In [5]:
t1 = time.time()
response = client.chat.completions.create(
          # model='gpt-4o-2024-08-06',
          # note: the timestamp indicates the date of the model to avoid changing 
          # behavior due to new models being released.
          # but: proprietory models might be taken offline and not be availible anymore at some point.  
          model = 'gpt-5.4-mini',
          messages=[
            {"role": "system", "content": myContext},
            {"role": "user", "content":  myPrompt}
          ], 
          store=False, 
        )  
dt = time.time() - t1
responseDict = response.to_dict(mode='json') # choose formatting

### Inspect the Response
The response contains, in addition to the output itself, also a lot of other information: which model was used, type of input, tokens created, etc.  
The output (string) is part of the choices. The tokens/second depend on the model and might fluctuate with the "mini" and "nano" models being faster than the larger ones. 


In [6]:
nTokens =  responseDict['usage']['prompt_tokens']
print('created {} tokens in {} seconds | {} tokens/second\n'.format(nTokens, round(dt, 1), round(nTokens/dt, 2)))

for key, value in dict(response).items(): 
        print(key, ': ', value, '\n')
    

created 2054 tokens in 6.8 seconds | 300.22 tokens/second

id :  chatcmpl-DnJSAXk52yvvv80j4wHMORP1DQtV9 

choices :  [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here is a compact Exudyn example that creates a 3-link rigid-body chain aligned along the X-axis, with:\n\n- each link: `m = 10 kg`, `L = 2 m`, `H = W = 0.1 m`\n- gravity acting in negative Y-direction\n- first link attached to ground by a revolute joint at its left end\n- a prismatic joint attached to the last link\n\n```python\nimport exudyn as exu\nfrom exudyn.utilities import *\nimport exudyn.graphics as graphics\nimport numpy as np\nfrom math import pi\n\n# -----------------------------\n# system setup\n# -----------------------------\nSC = exu.SystemContainer()\nmbs = SC.AddSystem()\n\n# ground\noGround = mbs.CreateGround()\n\n# -----------------------------\n# link parameters\n# -----------------------------\nm = 10.0\nL = 2.0\nH = 0.1\nW = 0.1\n\n# compute inertia from ma

Multiple ```choices``` could be part of the response, but in our case it should only be a single one.  

In [7]:
for i in range(len(response.choices)): 
    res = response.choices[i]
    print(res.message.content)
    strOutput = res.message.content

Here is a compact Exudyn example that creates a 3-link rigid-body chain aligned along the X-axis, with:

- each link: `m = 10 kg`, `L = 2 m`, `H = W = 0.1 m`
- gravity acting in negative Y-direction
- first link attached to ground by a revolute joint at its left end
- a prismatic joint attached to the last link

```python
import exudyn as exu
from exudyn.utilities import *
import exudyn.graphics as graphics
import numpy as np
from math import pi

# -----------------------------
# system setup
# -----------------------------
SC = exu.SystemContainer()
mbs = SC.AddSystem()

# ground
oGround = mbs.CreateGround()

# -----------------------------
# link parameters
# -----------------------------
m = 10.0
L = 2.0
H = 0.1
W = 0.1

# compute inertia from mass and dimensions
# (COM at center of each cuboid)
inertiaLink = InertiaCuboid(density=m/(L*H*W), sideLengths=[L, H, W])

# graphics for one link (local center at COM -> [0,0,0])
graphicsLink = graphics.Brick(centerPoint=[0, 0, 0],
         

### Extract Code
The model (usually) not only outputs the code itself, but also some explanation before and/or afterwards.  
The code itself should be put between the ```python``` tag by the model as shown below, although not all models might follow this convention. 


In [8]:
flagExtractedCode = False
try: 
    code = strOutput.split("```python")[1].split("```")[0]
    flagExtractedCode = True 
except: 
    print('Model did not use ```python to mark code. Trying "python3". ')

if not(flagExtractedCode): 
    try: 
        code = strOutput.split("python3")[1].split("--------------------------")[0]
        flagExtractedCode = True
    except: 
        print('Model did not use python3 mark either.')

CheckOutputLLM(code)

if flagExtractedCode: 
    exec(code)
else: 
    print('code could not be extracted from model output')


Python WARNING [file 'C:\Users\peter\anaconda3\envs\MLMUBO\Lib\site-packages\exudyn\solver.py', line 260]: 
CSolverImplicitSecondOrder::InitializeSolverInitialConditions: System Jacobian seems to be singular / not invertible!The solver returned the causing system equation number (coordinate number) = 41

******************************************


******************************************



SYSTEM ERROR [file 'C:\Users\peter\anaconda3\envs\MLMUBO\Lib\site-packages\exudyn\solver.py', line 260]: 
CSolverBase::Newton: System Jacobian seems to be singular / not invertible!
  time/load step #1, time = 0.002
  causing system equation number (coordinate number) = 41



SYSTEM ERROR [file 'C:\Users\peter\anaconda3\envs\MLMUBO\Lib\site-packages\exudyn\solver.py', line 260]: 
EXUDYN raised internal error in 'CSolverBase::SolveSteps':
Exudyn: parsing of Python file terminated due to system error


******************************
DYNAMIC SOLVER FAILED:
  use showHints=True to show helpful infor

ValueError: SolveDynamic terminated

Note: A common error of the LLM in this task is to _hallucinate_ color options which are not available, for example _skyblue_. 

### Task (15 min)
Recreate the previous oscillator example using scipy and the OpenAI API.  